# Day 4 — Expectation Suites

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/great-expectations-certified/notebooks/day-04-expectation-suites.ipynb)

**Badge:** Practice  
**Course:** Great Expectations for Data Quality

---

## What you will learn

- How GX stores Expectation Suites as JSON in the Expectations Store
- How to create, populate, save, and reload named suites
- How to auto-generate a draft suite from a sample DataFrame and tighten it
- How to version suites safely before overwriting
- How to round-trip a suite through JSON serialization

> **Tip:** Treat suite JSON files like migrations — name them semantically, commit to version control, never overwrite without archiving. A suite is your data contract.

## Install dependencies

In [ ]:
# Install Great Expectations (quiet mode for Colab)
%pip install great_expectations --quiet

## 1. Reading the Expectation Suite Management Guide

Before writing code, read the official guide on managing suites:
[Manage Expectation Suites](https://docs.greatexpectations.io/docs/core/define_expectations/manage_expectation_suites)

Key facts to keep in mind:

- Suites are stored in the **Expectations Store**, which defaults to a local `great_expectations/expectations/` directory.
- Each suite is a JSON file named after the suite (e.g., `orders.critical.json`).
- `context.suites.add(suite)` registers a new suite in the store.
- `context.suites.get(name)` retrieves an existing suite by name.
- `context.suites.save(suite)` persists changes back to the store.
- An `EphemeralDataContext` (from `gx.get_context()`) keeps everything in memory — ideal for notebooks and Colab.

In [ ]:
import great_expectations as gx
import pandas as pd
import json, copy, re, sys

# EphemeralDataContext — everything in memory, no filesystem writes required
context = gx.get_context()
print('GX version:', gx.__version__)
print('Context type:', type(context).__name__)

## 2. Generating a Synthetic Orders DataFrame

We will work with a synthetic `orders` dataset throughout this notebook. This avoids any external downloads and makes the notebook fully self-contained.

In [ ]:
import numpy as np

rng = np.random.default_rng(42)
N = 200

orders_df = pd.DataFrame({
    'order_id':    [f'ORD-{i:05d}' for i in range(1, N + 1)],
    'customer_id': rng.integers(1000, 9999, size=N),
    'amount':      rng.uniform(5.0, 500.0, size=N).round(2),
    'status':      rng.choice(['pending', 'shipped', 'delivered', 'cancelled'], size=N),
    'email':       [f'user{i}@example.com' for i in range(1, N + 1)],
})

print(orders_df.shape)
orders_df.head()

## 3. Creating Two Named Expectation Suites

We create two suites with distinct purposes:

- **`orders.critical`** — nullness, schema, and row-count expectations. These are hard gates; any failure blocks downstream processing.
- **`orders.distributions`** — value ranges and regex format checks. These catch drift in business meaning (e.g., amounts going negative) without necessarily blocking a pipeline.

See `gx.ExpectationSuite` and `context.suites.add()` in the [API reference](https://docs.greatexpectations.io/docs/reference/api/core/ExpectationSuite).

In [ ]:
# ---- Suite 1: orders.critical ----
suite_critical = context.suites.add(
    gx.ExpectationSuite(name='orders.critical')
)

# Row count: expect at least 50 rows and no more than 1 million
suite_critical.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(min_value=50, max_value=1_000_000)
)

# Schema: columns must exist
for col in ['order_id', 'customer_id', 'amount', 'status', 'email']:
    suite_critical.add_expectation(
        gx.expectations.ExpectColumnToExist(column=col)
    )

# Nullness: critical columns must have no nulls
for col in ['order_id', 'customer_id', 'amount']:
    suite_critical.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column=col)
    )

context.suites.save(suite_critical)
print(f'Saved suite: {suite_critical.name!r} with {len(suite_critical.expectations)} expectations')

In [ ]:
# ---- Suite 2: orders.distributions ----
suite_dist = context.suites.add(
    gx.ExpectationSuite(name='orders.distributions')
)

# Value range: amounts must be positive and under $10,000
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='amount', min_value=0.01, max_value=10_000.0
    )
)

# Categorical: status must be from a known set
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToBeInSet(
        column='status',
        value_set=['pending', 'shipped', 'delivered', 'cancelled']
    )
)

# Regex format: email must look like a real email
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column='email',
        regex=r'^[\w.+-]+@[\w-]+\.[a-zA-Z]{2,}$'
    )
)

# order_id must match pattern ORD-NNNNN
suite_dist.add_expectation(
    gx.expectations.ExpectColumnValuesToMatchRegex(
        column='order_id',
        regex=r'^ORD-\d{5}$'
    )
)

context.suites.save(suite_dist)
print(f'Saved suite: {suite_dist.name!r} with {len(suite_dist.expectations)} expectations')

## 4. Auto-Generating a Draft Suite from a Sample DataFrame

GX can profile a DataFrame and auto-generate expectations. The result is a draft that you must **tighten manually** — auto-profiling is generous by design; it will not catch subtle business constraints.

We simulate this workflow by programmatically building expectations derived from DataFrame statistics, then editing the thresholds.

In [ ]:
# Create a data source + asset + batch so we can use the Validator API
data_source = context.data_sources.add_pandas('orders_source')
data_asset  = data_source.add_dataframe_asset('orders_asset')
batch_def   = data_asset.add_batch_definition_whole_dataframe('orders_batch')
batch       = batch_def.get_batch(batch_parameters={'dataframe': orders_df})

print('Batch loaded, row count:', len(orders_df))

In [ ]:
# Auto-derive draft expectations from observed stats
# (simulates what a profiler would produce — generous thresholds)
draft_suite = context.suites.add(
    gx.ExpectationSuite(name='orders.draft')
)

observed_min = float(orders_df['amount'].min())
observed_max = float(orders_df['amount'].max())
observed_row_count = len(orders_df)

# Profiler would set wide bounds — 0 to 2x observed max
draft_suite.add_expectation(
    gx.expectations.ExpectColumnValuesToBeBetween(
        column='amount',
        min_value=0,
        max_value=round(observed_max * 2, 2)
    )
)
draft_suite.add_expectation(
    gx.expectations.ExpectTableRowCountToBeBetween(
        min_value=0,
        max_value=observed_row_count * 10
    )
)

context.suites.save(draft_suite)

print('Draft suite expectations (generous thresholds):')
for exp in draft_suite.expectations:
    print(' -', exp)

In [ ]:
# Tighten the draft: reload, inspect, update thresholds
# In production you would edit the JSON file directly; here we mutate in memory
draft_suite_loaded = context.suites.get('orders.draft')

for exp in draft_suite_loaded.expectations:
    if hasattr(exp, 'max_value') and exp.column == 'amount' if hasattr(exp, 'column') else False:
        exp.max_value = 500.0   # tighten: domain knowledge says max order is $500
        exp.min_value = 1.0     # tighten: no free orders

context.suites.save(draft_suite_loaded)
print('Tightened draft suite expectations:')
for exp in draft_suite_loaded.expectations:
    print(' -', exp)

## 5. Loading an Existing Suite and Adding Expectations

Use `context.suites.get(name)` to retrieve a saved suite, modify it, and call `context.suites.save()` to persist changes. This is the correct update pattern — never construct a new suite with the same name.

In [ ]:
# Reload orders.critical and add two new expectations
suite_to_update = context.suites.get('orders.critical')
print(f'Loaded suite: {suite_to_update.name!r}')
print(f'Existing expectations: {len(suite_to_update.expectations)}')

# New expectation 1: customer_id must be an integer-like value (no strings)
suite_to_update.add_expectation(
    gx.expectations.ExpectColumnValuesToBeOfType(
        column='customer_id',
        type_='int64'
    )
)

# New expectation 2: column count must equal exactly 5
suite_to_update.add_expectation(
    gx.expectations.ExpectTableColumnCountToEqual(value=5)
)

context.suites.save(suite_to_update)
print(f'Updated suite now has {len(suite_to_update.expectations)} expectations')

# Confirm by re-fetching
confirmed = context.suites.get('orders.critical')
print(f'Confirmed from store: {len(confirmed.expectations)} expectations')

## 6. Suite Versioning Helper

Before overwriting a suite, archive the previous version. This is a lightweight pattern that does not require a database — it copies the serialized suite JSON to an in-memory registry keyed by `(suite_name, tag)`.

In production you would write the JSON to a versioned file path (e.g., `expectations_archive/orders.critical_v1.json`) or push to a Git tag.

In [ ]:
# In-memory archive: {(suite_name, tag): suite_dict}
_suite_archive: dict = {}

def version_suite(suite_name: str, tag: str, context=context) -> None:
    """Archive a copy of suite JSON before overwriting.

    Args:
        suite_name: Name of the suite in the Expectations Store.
        tag:        Version label, e.g. 'v1' or '2026-06-14'.
        context:    GX DataContext (defaults to the notebook context).

    Raises:
        KeyError: If suite_name does not exist in the store.
    """
    suite = context.suites.get(suite_name)
    # Serialize to dict via JSON round-trip
    suite_dict = json.loads(suite.json())
    archive_key = (suite_name, tag)
    if archive_key in _suite_archive:
        print(f'Warning: overwriting existing archive for {archive_key}')
    _suite_archive[archive_key] = suite_dict
    print(f'Archived {suite_name!r} as tag {tag!r} ({len(suite_dict["expectations"])} expectations)')


def list_versions(suite_name: str) -> list:
    """List all archived tags for a given suite name."""
    return [tag for (name, tag) in _suite_archive if name == suite_name]


# Demonstrate versioning
version_suite('orders.critical', 'v1')

# Now safely modify and overwrite
suite_v2 = context.suites.get('orders.critical')
suite_v2.add_expectation(
    gx.expectations.ExpectColumnValuesToNotBeNull(column='status')
)
context.suites.save(suite_v2)

version_suite('orders.critical', 'v2')
print('Available versions:', list_versions('orders.critical'))

## 7. JSON Serialization Round-Trip

According to the [ExpectationSuite API reference](https://docs.greatexpectations.io/docs/reference/api/core/ExpectationSuite), `suite.json()` serializes to a JSON string. You can deserialize using `gx.ExpectationSuite.parse_raw()` (Pydantic v1) or `gx.ExpectationSuite.model_validate_json()` (Pydantic v2).

The test below verifies that:
1. The serialized JSON is valid and parseable.
2. The deserialized suite has the same name and expectation count.
3. Individual expectation types are preserved.

In [ ]:
def test_suite_json_round_trip(suite: gx.ExpectationSuite) -> None:
    """Assert that a suite survives a JSON round-trip intact."""
    # Serialize
    suite_json_str = suite.json()
    assert isinstance(suite_json_str, str), 'json() must return a string'

    # Parse to dict to validate JSON structure
    suite_dict = json.loads(suite_json_str)
    assert suite_dict['name'] == suite.name, 'Name mismatch after serialization'
    assert len(suite_dict['expectations']) == len(suite.expectations), \
        f'Expectation count mismatch: {len(suite_dict["expectations"])} vs {len(suite.expectations)}'

    # Verify each expectation type is preserved
    original_types = [type(e).__name__ for e in suite.expectations]
    serial_types   = [e['type'] for e in suite_dict['expectations']]
    assert original_types == serial_types, f'Type mismatch: {original_types} vs {serial_types}'

    print(f'PASS: suite {suite.name!r} round-trip OK ({len(suite.expectations)} expectations)')


# Run the test on both suites
test_suite_json_round_trip(context.suites.get('orders.critical'))
test_suite_json_round_trip(context.suites.get('orders.distributions'))

## 8. Running Validations Against the Suites

With suites defined, we can validate our DataFrame batch. We use `ValidationDefinition` to bind a suite to a batch, then call `.run()`. This previews the Checkpoint pattern you will use in Day 5.

In [ ]:
# Validate against orders.critical
validation_def = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='validate_orders_critical',
        data=batch_def,
        suite=context.suites.get('orders.critical'),
    )
)

result = validation_def.run(batch_parameters={'dataframe': orders_df})
print('Validation success:', result.success)
print('Expectations evaluated:', len(result.results))
failures = [r for r in result.results if not r.success]
print('Failures:', len(failures))
for f in failures:
    print(' FAIL:', f.expectation_config.type)

In [ ]:
# Validate against orders.distributions
validation_def_dist = context.validation_definitions.add(
    gx.ValidationDefinition(
        name='validate_orders_distributions',
        data=batch_def,
        suite=context.suites.get('orders.distributions'),
    )
)

result_dist = validation_def_dist.run(batch_parameters={'dataframe': orders_df})
print('Validation success:', result_dist.success)
failures_dist = [r for r in result_dist.results if not r.success]
print('Failures:', len(failures_dist))
for f in failures_dist:
    print(' FAIL:', f.expectation_config.type, '|', f.result)

## Challenge — Demonstrate a Validation Failure

Inject a bad row into the orders DataFrame and show that `orders.critical` catches it.

Specifically:
1. Add a row where `amount` is `None` (null).
2. Re-run the `orders.critical` validation.
3. Print the expectation that failed and the unexpected count.
4. (Bonus) Also inject an invalid `email` format and confirm `orders.distributions` catches it.

In [ ]:
# ---- Challenge solution ----

# 1. Inject a bad row (null amount)
bad_row = pd.DataFrame([{
    'order_id':    'ORD-99999',
    'customer_id': 1234,
    'amount':      None,          # <-- null: violates ExpectColumnValuesToNotBeNull
    'status':      'pending',
    'email':       'bad-email',   # <-- also invalid: violates email regex
}])
dirty_df = pd.concat([orders_df, bad_row], ignore_index=True)
print(f'Dirty DataFrame rows: {len(dirty_df)}  (clean: {len(orders_df)})')

# 2. Run critical validation on dirty data
result_dirty = validation_def.run(batch_parameters={'dataframe': dirty_df})
print('\norders.critical on dirty data — success:', result_dirty.success)

# 3. Show which expectations failed
print('\nFailed expectations:')
for r in result_dirty.results:
    if not r.success:
        exp_type = r.expectation_config.type
        col = r.expectation_config.kwargs.get('column', 'TABLE')
        print(f'  [{col}] {exp_type}')
        if 'unexpected_count' in r.result:
            print(f'    unexpected_count = {r.result["unexpected_count"]}')

# 4. Bonus: check distributions suite catches bad email
result_dist_dirty = validation_def_dist.run(batch_parameters={'dataframe': dirty_df})
print('\norders.distributions on dirty data — success:', result_dist_dirty.success)
for r in result_dist_dirty.results:
    if not r.success:
        print(f'  [{r.expectation_config.kwargs.get("column", "TABLE")}] {r.expectation_config.type}')
        if 'unexpected_list' in r.result:
            print(f'    unexpected_list sample: {r.result["unexpected_list"][:3]}')

## Recap

| Concept | Key API |
|---|---|
| Create a suite | `gx.ExpectationSuite(name=...)` |
| Register in store | `context.suites.add(suite)` |
| Add an expectation | `suite.add_expectation(gx.expectations.Expect...(...))` |
| Persist changes | `context.suites.save(suite)` |
| Reload from store | `context.suites.get('name')` |
| Serialize to JSON | `suite.json()` |
| Validate a batch | `ValidationDefinition.run(batch_parameters=...)` |

**Design principles learned today:**

- Split suites by *severity*: critical suites block pipelines; distribution suites send alerts.
- Auto-profiling gives you a draft — always tighten thresholds to match domain knowledge.
- Archive before overwrite: the `version_suite()` helper is a lightweight contract-change log.
- JSON round-trip testing is fast and catches serialization regressions early.

**Next up — Day 5:** Wiring suites into Checkpoints and generating browsable Data Docs HTML reports.